In [4]:
from statistics import mean
from datetime import datetime

from sqlalchemy import select
from sqlalchemy.orm import Session, lazyload, joinedload
from sqlalchemy import create_engine
from collections import defaultdict
from dataclasses import dataclass
import json

from sqlalchemy import and_

import pandas as pd

import toml
import datetime

import matplotlib.pyplot as plt

from ss13tools.settings import make_engine
from ss13tools.model import Round, Feedback, LegacyPopulation

import matplotlib.dates as pltd
import numpy as np
from sqlalchemy import desc

engine = make_engine("settings.toml")
session = Session(engine)

In [3]:
session.query(Round).filter(Round.start_datetime >= datetime.datetime(2023, 1, 17)).order_by(desc(Round.start_datetime)).count()

4562

In [3]:
session.query(Round).filter(Round.start_datetime >= datetime.datetime(2023, 1, 17)).order_by(desc(Round.start_datetime)).first()

<Round#41292 [2024-08-01] extended/Cyberiad>

In [2]:
session.query(Round).filter(Round.start_datetime >= datetime.datetime(2023, 1, 17)).order_by(desc(-Round.start_datetime))

In [5]:
@dataclass(frozen=True)
class RoundPopSummary:
    previous_round: int
    current_round: int
    next_round: int
    earliest_pop: int
    latest_pop: int
    current_map: str
    next_map: str
    next_earliest_pop: int
    start_time: datetime

In [33]:
count = 0
round_pop_summaries = dict()
for round in session.query(Round).filter(Round.start_datetime >= datetime.datetime(2023, 1, 17)).order_by(desc(Round.start_datetime)).all():
    count += 1
    if count % 20 == 0:
        print(round.id)
    next_round = None
    next_round_id = round.id+1
    next_map = ""
    next_earliest_pop = -1
    if next_round_id in round_pop_summaries:
        next_map = round_pop_summaries[next_round_id].current_map
        next_earliest_pop = round_pop_summaries[next_round_id].earliest_pop
    else:
        next_round = session.get(Round, next_round_id)
        if not next_round:
            continue
        next_map = next_round.map_name
        next_earliest_pop = next_round.populations[0]
    summary = RoundPopSummary(
        previous_round=round.id - 1,
        current_round=round.id,
        next_round=round.id + 1,
        earliest_pop=round.populations[0],
        latest_pop=round.populations[-1],
        current_map=round.map_name,
        next_map=next_map,
        next_earliest_pop=next_earliest_pop,
        start_time=round.start_datetime)
    round_pop_summaries[round.id] = summary

41273
41253
41233
41213
41193
41172
41152
41132
41112
41092
41072
41052
41032
41012
40992
40971
40951
40931
40911
40891
40871
40851
40831
40809
40789
40769
40749
40729
40709
40689
40669
40649
40629
40609
40589
40569
40549
40529
40509
40489
40469
40449
40429
40409
40389
40369
40349
40329
40309
40277
40257
40237
40217
40197
40177
40157
40136
40116
40096
40076
40056
40036
40016
39996
39976
39955
39935
39915
39895
39875
39855
39835
39815
39795
39775
39755
39734
39714
39694
39674
39654
39634
39614
39594
39574
39554
39533
39513
39493
39473
39453
39433
39413
39393
39373
39352
39332
39312
39292
39272
39252
39232
39212
39191
39171
39151
39131
39111
39091
39071
39051
39031
39011
38991
38971
38951
38931
38911
38891
38871
38851
38831
38810
38790
38770
38750
38730
38710
38690
38670
38650
38630
38610
38590
38570
38550
38530
38510
38490
38468
38447
38427
38407
38387
38367
38347
38327
38306
38286
38265
38245
38225
38205
38185
38165
38145
38125
38105
38085
38065
38045
38024
38004
37984
37964
37944
3792

In [7]:
len(round_pop_summaries)

4521

In [8]:
round_pop_summaries[40000]

RoundPopSummary(previous_round=39999, current_round=40000, next_round=40001, earliest_pop=<Players  76@2024-04-20 15:48:30>, latest_pop=<Players  99@2024-04-20 17:09:12>, current_map='Delta', next_map='Cyberiad', next_earliest_pop=<Players  85@2024-04-20 17:11:10>, start_time=datetime.datetime(2024, 4, 20, 15, 52, 35))

In [9]:
class MonthPopTransitionSummary:
    def __init__(self, dt):
        self.dt = dt
        self.map_to_map = defaultdict(list)


In [19]:
summaries = dict()
mapnames = set()
for id, summary in round_pop_summaries.items():
    month = summary.start_time.strftime("%Y-%m")
    if month not in summaries:
        summaries[month] = MonthPopTransitionSummary(month)
    transition_summary = summaries[month]
    transition_summary.map_to_map[(summary.current_map, summary.next_map)].append(summary.next_earliest_pop.playercount - summary.latest_pop.playercount)
    mapnames.add(summary.current_map)
    mapnames.add(summary.next_map)


In [20]:
mapnames

{'CereStation', 'Cyberiad', 'Delta', 'MetaStation'}

In [15]:
summaries

{'2024-08': <__main__.MonthPopTransitionSummary at 0x1fd6b84cb50>,
 '2024-07': <__main__.MonthPopTransitionSummary at 0x1fd2785bbb0>,
 '2024-06': <__main__.MonthPopTransitionSummary at 0x1fd278580d0>,
 '2024-05': <__main__.MonthPopTransitionSummary at 0x1fd2785ac50>,
 '2024-04': <__main__.MonthPopTransitionSummary at 0x1fd6b90ca60>,
 '2024-03': <__main__.MonthPopTransitionSummary at 0x1fd27859930>,
 '2024-02': <__main__.MonthPopTransitionSummary at 0x1fd27858250>,
 '2024-01': <__main__.MonthPopTransitionSummary at 0x1fd278596c0>,
 '2023-12': <__main__.MonthPopTransitionSummary at 0x1fd27859e10>,
 '2023-11': <__main__.MonthPopTransitionSummary at 0x1fd2785a3e0>,
 '2023-10': <__main__.MonthPopTransitionSummary at 0x1fd27859d50>,
 '2023-09': <__main__.MonthPopTransitionSummary at 0x1fd2785be80>,
 '2023-08': <__main__.MonthPopTransitionSummary at 0x1fd2785b310>,
 '2023-07': <__main__.MonthPopTransitionSummary at 0x1fd27858310>}

In [17]:
summaries['2024-07'].map_to_map

defaultdict(list,
            {('Cyberiad', 'Cyberiad'): [-12,
              -16,
              -4,
              -7,
              -11,
              -11,
              -5,
              -7,
              -4,
              -20,
              -9,
              -16,
              -3,
              -15,
              -3,
              -6,
              -7,
              -8,
              -14,
              -11,
              -18,
              -15,
              -14,
              -14,
              -7,
              -10,
              -5,
              -11,
              -7,
              -10,
              -18,
              -17,
              -26,
              -10,
              -21,
              -10,
              -16,
              -1,
              4,
              -12,
              -15,
              -7,
              -8,
              -16,
              -13,
              -22,
              -17,
              -17,
              -11,
              -6,
              -18,
       

In [22]:
all_maps_to = dict()
for to_map in mapnames:
    all_maps_to[to_map] = dict()
    for from_map in mapnames:
        all_maps_to[to_map][from_map] = defaultdict(list)

for id, summary in round_pop_summaries.items():
    month = summary.start_time.strftime("%Y-%m")
    from_map = summary.current_map
    to_map = summary.next_map
    all_maps_to[to_map][from_map][month].append(summary.next_earliest_pop.playercount - summary.latest_pop.playercount)

In [32]:
all_maps_to


for to_map, from_maps in all_maps_to.items():
    first_row = [to_map]
    data_rows = []
    months = set()
    for from_map, monthdata in from_maps.items():
        months.update(sorted(monthdata.keys()))
    sorted_months = sorted(months)
    for month in sorted_months:
        first_row.append(month)
    for from_map, monthdata in from_maps.items():
        row = [from_map]
        for month in sorted_months:
            if month not in monthdata:
                row.append("-")
                continue
            row.append(str(int(mean(monthdata[month]))))
        data_rows.append(row)
    print(','.join(first_row))
    for row in data_rows:
        print(','.join(row))

Delta,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08
Delta,-11,-11,-24,-23,-22,-17,-19,-14,-6,-8,-11,-,-,-
CereStation,-13,-11,-16,-22,-16,-21,-18,-16,-13,-11,-11,-11,-8,-
MetaStation,-10,-12,-18,-23,-16,-13,-13,-21,-14,-10,-17,-8,-7,-11
Cyberiad,-17,-17,-22,-24,-27,-19,-18,-17,-14,-14,-14,-11,-11,-12
CereStation,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08
Delta,-8,-14,-24,-19,-14,-20,-15,-11,-17,-18,-18,-9,-13,-8
CereStation,-,-9,-18,-19,-17,-19,-15,-17,-13,-9,-18,-,-,-
MetaStation,-8,-15,-18,-22,-17,-16,-24,-15,-10,-16,-16,-15,-14,-11
Cyberiad,-13,-16,-17,-22,-24,-22,-19,-18,-15,-13,-15,-16,-14,-13
MetaStation,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08
Delta,-5,-9,-13,-19,-26,-15,-15,-15,-13,-11,-15,-9,-9,-
CereStation,-4,-17,-9,-16,-19,-18,-18,-11,-13,-5,-17,-9,-12,-5
MetaStation,-,-1

In [30]:
data_rows

[['Delta', -14, -16, -20, -20, -22, -19, -19, -18, -15, -14, -12, -9, -10, -5],
 ['CereStation',
  -9,
  -13,
  -19,
  -20,
  -19,
  -18,
  -14,
  -13,
  -17,
  -10,
  -15,
  -12,
  -11,
  -9],
 ['MetaStation',
  -12,
  -13,
  -15,
  -21,
  -20,
  -17,
  -16,
  -11,
  -10,
  -15,
  -16,
  -11,
  -10,
  '-'],
 ['Cyberiad',
  -19,
  -16,
  -17,
  -22,
  -22,
  -19,
  -17,
  -15,
  -14,
  -13,
  -15,
  -12,
  -12,
  '-']]

In [7]:
round = session.query(Round).filter(Round.start_datetime >= datetime.datetime(2023, 1, 17)).order_by(desc(-Round.start_datetime)).first()

In [10]:
round.populations[-1]

<Players  85@2023-07-24 11:04:57>

In [41]:
round = session.get(Round, 41000)

In [43]:
round.start_datetime

datetime.datetime(2024, 7, 9, 14, 29, 18)

In [44]:
round.end_datetime

datetime.datetime(2024, 7, 9, 16, 39, 53)

In [45]:
round.populations[0]

<Players  39@2024-07-09 14:25:15>

In [46]:
round.populations[-1]

<Players  67@2024-07-09 16:35:39>

In [40]:
session = Session(engine)